In [ ]:
import os
import math
import json
import random
from glob import glob

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- paths ----------
CLEAN_DIR = r"D:\Masters\Speech\Embeddings\clean"   # change this
PAIR_DIR = r"D:\Masters\Speech\pause_rhythm_pairs"
MODEL_PATH = r"D:\Masters\Speech\reverse_pause_rhythm_best.pt"

# ---------- forward diffusion ----------
T_STEPS = 30
SCHEDULE = "cosine"

PAUSE_RHYTHM_CFG = {
    "pause_density_base": 0.015,
    "pause_density_scale": 0.06,
    "pause_len_min": 3,
    "pause_len_max": 10,

    "rhythm_density_base": 0.020,
    "rhythm_density_scale": 0.08,
    "rhythm_len_min": 4,
    "rhythm_len_max": 14,

    "boundary_power": 1.4,
    "pause_flatten_strength": 0.65,
    "pause_weaken_strength": 0.22,
    "pause_smooth_kernel": 9,

    "rhythm_reindex_strength": 0.35,
    "rhythm_local_mix_strength": 0.30,
    "rhythm_shift_choices": [-3, -2, -1, 1, 2, 3],

    "global_timing_jitter": 0.08,
}

In [ ]:
def moving_average_1d(x: np.ndarray, kernel_size: int = 5) -> np.ndarray:
    """
    Moving average along time axis for shape (T, D).
    """
    if kernel_size <= 1:
        return x.astype(np.float32)

    x = x.astype(np.float32)
    T, D = x.shape
    pad = kernel_size // 2
    x_pad = np.pad(x, ((pad, pad), (0, 0)), mode="edge")

    out = np.zeros_like(x, dtype=np.float32)
    for t in range(T):
        out[t] = np.mean(x_pad[t:t + kernel_size], axis=0)
    return out.astype(np.float32)


def get_step_weights(T: int, schedule: str = "cosine") -> np.ndarray:
    if schedule == "uniform":
        w = np.ones(T, dtype=np.float32)
    elif schedule == "linear":
        w = np.arange(1, T + 1, dtype=np.float32)
    elif schedule == "cosine":
        t = np.linspace(0, 1, T, dtype=np.float32)
        w = 1.0 - np.cos((t + 1e-8) * np.pi / 2.0)
    else:
        raise ValueError(f"Unknown schedule: {schedule}")

    w = np.maximum(w, 1e-8)
    w = w / np.sum(w)
    return w.astype(np.float32)


def cosine_similarity_mean(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a = a.astype(np.float32)
    b = b.astype(np.float32)
    num = np.sum(a * b, axis=1)
    den = np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + eps
    return float(np.mean(num / den))


def evaluate_pair(clean: np.ndarray, corrupted: np.ndarray) -> dict:
    diff = corrupted - clean
    return {
        "mean_abs_error": float(np.mean(np.abs(diff))),
        "max_abs_error": float(np.max(np.abs(diff))),
        "mean_l2_per_frame": float(np.mean(np.linalg.norm(diff, axis=1))),
        "cosine": cosine_similarity_mean(clean, corrupted),
    }

In [ ]:
def sample_spans_from_prob(
    Tseq: int,
    prob: np.ndarray,
    num_spans: int,
    span_len_min: int,
    span_len_max: int,
    rng: np.random.Generator
):
    spans = []
    for _ in range(num_spans):
        center = int(rng.choice(Tseq, p=prob))
        span_len = int(rng.integers(span_len_min, span_len_max + 1))
        start = max(0, center - span_len // 2)
        end = min(Tseq, start + span_len)
        spans.append((start, end))
    return spans


def spans_to_mask(Tseq: int, spans):
    mask = np.zeros(Tseq, dtype=np.float32)
    for s, e in spans:
        mask[s:e] = 1.0
    return mask

In [ ]:
def pause_rhythm_forward_step_pronounced(
    x_prev: np.ndarray,
    step_weight: float,
    pause_density_base: float = 0.015,
    pause_density_scale: float = 0.06,
    pause_len_min: int = 3,
    pause_len_max: int = 10,
    rhythm_density_base: float = 0.020,
    rhythm_density_scale: float = 0.08,
    rhythm_len_min: int = 4,
    rhythm_len_max: int = 14,
    boundary_power: float = 1.4,
    pause_flatten_strength: float = 0.65,
    pause_weaken_strength: float = 0.22,
    pause_smooth_kernel: int = 9,
    rhythm_reindex_strength: float = 0.35,
    rhythm_local_mix_strength: float = 0.30,
    rhythm_shift_choices = (-3, -2, -1, 1, 2, 3),
    global_timing_jitter: float = 0.08,
    seed: int = 42
):
    """
    Fixed-length pause + rhythm corruption.

    Pause effect:
      - flatten selected spans toward local mean
      - reduce local variation / make them plateau-like

    Rhythm effect:
      - re-index local spans irregularly
      - mix nearby shifted frames to break smooth cadence
      - add small global timing jitter
    """
    rng = np.random.default_rng(seed)

    x_prev = x_prev.astype(np.float32)
    Tseq, D = x_prev.shape
    x_t = x_prev.copy()

    # ------------------------------------------------------------
    # 1) Estimate boundary / phrase-likelihood regions
    # ------------------------------------------------------------
    frame_diff = np.linalg.norm(x_prev[1:] - x_prev[:-1], axis=1)
    frame_diff = np.pad(frame_diff, (1, 0), mode="edge").astype(np.float32)

    boundary_score = np.power(frame_diff + 1e-8, boundary_power)
    boundary_prob = boundary_score / np.sum(boundary_score)

    low_transition_score = 1.0 / (frame_diff + 1e-4)
    low_transition_prob = low_transition_score / np.sum(low_transition_score)

    # ------------------------------------------------------------
    # 2) Pause-like spans
    # pause often near weak boundaries / chunking irregularities,
    # so blend boundary and low-transition probabilities
    # ------------------------------------------------------------
    pause_prob = 0.55 * boundary_prob + 0.45 * low_transition_prob
    pause_prob = pause_prob / np.sum(pause_prob)

    num_pause_spans = max(1, int((pause_density_base + pause_density_scale * step_weight) * Tseq))
    pause_spans = sample_spans_from_prob(
        Tseq=Tseq,
        prob=pause_prob,
        num_spans=num_pause_spans,
        span_len_min=pause_len_min,
        span_len_max=pause_len_max,
        rng=rng
    )
    pause_mask = spans_to_mask(Tseq, pause_spans)[:, None]

    # local mean / plateau creation
    x_smooth = moving_average_1d(x_prev, kernel_size=pause_smooth_kernel)
    flatten_alpha = pause_flatten_strength * (0.35 + 0.65 * step_weight)
    weaken_alpha = pause_weaken_strength * (0.35 + 0.65 * step_weight)

    # flatten toward smoothed local trajectory
    x_t = x_t + pause_mask * (flatten_alpha * (x_smooth - x_t))

    # weaken magnitude slightly to mimic pause-like reduced activity
    x_t = x_t * (1.0 - pause_mask * weaken_alpha)

    # ------------------------------------------------------------
    # 3) Rhythm-warp spans
    # use low-transition regions a bit more, since rhythm acts on flow
    # ------------------------------------------------------------
    rhythm_prob = 0.35 * boundary_prob + 0.65 * low_transition_prob
    rhythm_prob = rhythm_prob / np.sum(rhythm_prob)

    num_rhythm_spans = max(1, int((rhythm_density_base + rhythm_density_scale * step_weight) * Tseq))
    rhythm_spans = sample_spans_from_prob(
        Tseq=Tseq,
        prob=rhythm_prob,
        num_spans=num_rhythm_spans,
        span_len_min=rhythm_len_min,
        span_len_max=rhythm_len_max,
        rng=rng
    )
    rhythm_mask = spans_to_mask(Tseq, rhythm_spans)[:, None]

    reindex_alpha = rhythm_reindex_strength * (0.35 + 0.65 * step_weight)
    local_mix_alpha = rhythm_local_mix_strength * (0.35 + 0.65 * step_weight)

    # local reindexing inside each rhythm span
    for (s, e) in rhythm_spans:
        seg = x_t[s:e].copy()
        L = e - s
        if L <= 2:
            continue

        # irregular time mapping inside fixed length
        idx = np.linspace(0, L - 1, L).astype(np.float32)

        noise = rng.normal(0.0, 1.0, size=L).astype(np.float32)
        noise = moving_average_1d(noise[:, None], kernel_size=min(5, L if L % 2 == 1 else max(1, L-1))).squeeze(1)

        idx_warp = idx + reindex_alpha * noise * max(1.0, L / 6.0)
        idx_warp = np.clip(idx_warp, 0, L - 1)

        idx0 = np.floor(idx_warp).astype(np.int32)
        idx1 = np.clip(idx0 + 1, 0, L - 1)
        frac = (idx_warp - idx0)[:, None]

        warped = (1.0 - frac) * seg[idx0] + frac * seg[idx1]
        x_t[s:e] = warped.astype(np.float32)

    # local cadence-breaking mix with shifted frames
    shift = int(rng.choice(rhythm_shift_choices))
    x_shift = np.roll(x_prev, shift=shift, axis=0)
    x_shift_smooth = moving_average_1d(x_shift, kernel_size=5)
    x_ref = 0.5 * x_shift + 0.5 * x_shift_smooth
    x_t = x_t + rhythm_mask * (local_mix_alpha * (x_ref - x_t))

    # ------------------------------------------------------------
    # 4) Small global timing jitter
    # ------------------------------------------------------------
    jitter_alpha = global_timing_jitter * (0.35 + 0.65 * step_weight)

    raw = rng.normal(0.0, 1.0, size=(Tseq, 1)).astype(np.float32)
    smooth = moving_average_1d(raw, kernel_size=7)
    x_global_ref = moving_average_1d(np.roll(x_prev, shift=1, axis=0), kernel_size=5)

    x_t = x_t + jitter_alpha * smooth * (x_global_ref - x_t)

    delta_t = x_t - x_prev

    meta = {
        "step_weight": float(step_weight),
        "num_pause_spans": int(num_pause_spans),
        "num_rhythm_spans": int(num_rhythm_spans),
        "pause_fraction": float(np.mean(pause_mask)),
        "rhythm_fraction": float(np.mean(rhythm_mask)),
        "delta_mean_l2": float(np.mean(np.linalg.norm(delta_t, axis=1))),
        "delta_max_l2": float(np.max(np.linalg.norm(delta_t, axis=1))),
        "flatten_alpha": float(flatten_alpha),
        "weaken_alpha": float(weaken_alpha),
        "reindex_alpha": float(reindex_alpha),
        "local_mix_alpha": float(local_mix_alpha),
        "jitter_alpha": float(jitter_alpha),
    }

    return x_t.astype(np.float32), delta_t.astype(np.float32), meta

In [ ]:
def pause_rhythm_forward_diffusion_chain_pronounced(
    x_clean: np.ndarray,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **cfg
):
    x_clean = x_clean.astype(np.float32)

    x_states = [x_clean.copy()]
    deltas = [None]
    metas = [None]

    x = x_clean.copy()
    step_weights = get_step_weights(T, schedule=schedule)

    for t in range(1, T + 1):
        x, delta_t, meta_t = pause_rhythm_forward_step_pronounced(
            x_prev=x,
            step_weight=float(step_weights[t - 1]),
            seed=seed + t,
            **cfg
        )
        meta_t["t"] = t
        x_states.append(x)
        deltas.append(delta_t)
        metas.append(meta_t)

    return x_states, deltas, metas

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
print("Found:", len(sample_files))

x_clean = np.load(sample_files[0]).astype(np.float32)

x_states, deltas, metas = pause_rhythm_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PAUSE_RHYTHM_CFG
)

x_T = x_states[-1]

print("clean -> x_T:", evaluate_pair(x_clean, x_T))
print("last meta:", metas[-1])

In [ ]:
def make_reverse_pause_rhythm_pairs_from_clean_folder(
    clean_dir: str,
    pair_dir: str,
    T: int = 30,
    schedule: str = "cosine",
    seed: int = 42,
    **cfg
):
    os.makedirs(pair_dir, exist_ok=True)

    files = sorted(glob(os.path.join(clean_dir, "*.npy")))
    print("Number of clean files:", len(files))

    manifest = []

    for i, fpath in enumerate(files, 1):
        x_clean = np.load(fpath).astype(np.float32)

        x_states, deltas, metas = pause_rhythm_forward_diffusion_chain_pronounced(
            x_clean=x_clean,
            T=T,
            schedule=schedule,
            seed=seed + i,
            **cfg
        )

        base = os.path.splitext(os.path.basename(fpath))[0]

        for t in range(1, T + 1):
            save_path = os.path.join(pair_dir, f"{base}_t{t:02d}.npz")
            np.savez_compressed(
                save_path,
                x_t=x_states[t].astype(np.float32),
                x_prev=x_states[t - 1].astype(np.float32),
                t=np.array([t], dtype=np.int64),
                T=np.array([T], dtype=np.int64),
            )
            manifest.append(save_path)

        summary = evaluate_pair(x_clean, x_states[-1])
        print(
            f"[{i:04d}/{len(files):04d}] {base} | "
            f"clean->x_T cosine={summary['cosine']:.6f} | "
            f"mae={summary['mean_abs_error']:.6f}"
        )

    manifest_path = os.path.join(pair_dir, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)

    print("Saved manifest to:", manifest_path)
    print("Total training pairs:", len(manifest))

In [ ]:
make_reverse_pause_rhythm_pairs_from_clean_folder(
    clean_dir=CLEAN_DIR,
    pair_dir=PAIR_DIR,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PAUSE_RHYTHM_CFG
)

In [ ]:
class ReversePauseRhythmDataset(Dataset):
    def __init__(self, pair_dir: str):
        manifest_path = os.path.join(pair_dir, "manifest.json")
        with open(manifest_path, "r") as f:
            self.files = json.load(f)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = np.load(self.files[idx])
        x_t = item["x_t"].astype(np.float32)
        x_prev = item["x_prev"].astype(np.float32)
        t = int(item["t"][0])
        T = int(item["T"][0])

        return {
            "x_t": x_t,
            "x_prev": x_prev,
            "t": t,
            "T": T,
            "length": x_t.shape[0]
        }


def collate_reverse(batch):
    max_len = max(x["length"] for x in batch)
    D = batch[0]["x_t"].shape[1]

    x_t_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    x_prev_batch = np.zeros((len(batch), max_len, D), dtype=np.float32)
    mask_batch = np.zeros((len(batch), max_len), dtype=np.float32)
    t_batch = np.zeros((len(batch),), dtype=np.int64)
    T_batch = np.zeros((len(batch),), dtype=np.int64)

    for i, item in enumerate(batch):
        L = item["length"]
        x_t_batch[i, :L] = item["x_t"]
        x_prev_batch[i, :L] = item["x_prev"]
        mask_batch[i, :L] = 1.0
        t_batch[i] = item["t"]
        T_batch[i] = item["T"]

    return {
        "x_t": torch.from_numpy(x_t_batch),
        "x_prev": torch.from_numpy(x_prev_batch),
        "mask": torch.from_numpy(mask_batch),
        "t": torch.from_numpy(t_batch),
        "T": torch.from_numpy(T_batch),
    }

In [ ]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t_norm: torch.Tensor):
        device = t_norm.device
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(0, half, device=device).float() / max(half - 1, 1)
        )
        args = t_norm[:, None] * freqs[None, :]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        if self.dim % 2 == 1:
            emb = F.pad(emb, (0, 1))
        return emb


class ResidualBlock1D(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 5, dilation: int = 1):
        super().__init__()
        pad = (kernel_size - 1) // 2 * dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=pad, dilation=dilation)
        self.norm1 = nn.GroupNorm(8, channels)
        self.norm2 = nn.GroupNorm(8, channels)

    def forward(self, x):
        res = x
        x = self.conv1(x)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.conv2(x)
        x = self.norm2(x)
        x = F.gelu(x)
        return x + res


class ReversePauseRhythmUNetLite(nn.Module):
    def __init__(self, dim_in=1024, hidden=384, time_dim=128):
        super().__init__()

        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, hidden)
        )

        self.in_proj = nn.Conv1d(dim_in, hidden, kernel_size=1)

        self.blocks = nn.ModuleList([
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
            ResidualBlock1D(hidden, kernel_size=5, dilation=1),
            ResidualBlock1D(hidden, kernel_size=5, dilation=2),
            ResidualBlock1D(hidden, kernel_size=5, dilation=4),
        ])

        self.out_proj = nn.Conv1d(hidden, dim_in, kernel_size=1)

    def forward(self, x_t, t, T):
        t_norm = t.float() / T.float().clamp(min=1.0)

        t_emb = self.time_embed(t_norm)
        t_emb = self.time_mlp(t_emb)

        x = x_t.transpose(1, 2)
        x = self.in_proj(x)
        x = x + t_emb[:, :, None]

        for blk in self.blocks:
            x = blk(x)

        delta_hat = self.out_proj(x).transpose(1, 2)
        x_prev_hat = x_t + delta_hat
        return x_prev_hat

In [ ]:
def masked_l1_l2_loss(pred, target, mask, l1_weight=1.0, l2_weight=1.0):
    mask = mask.unsqueeze(-1)

    l1 = torch.abs(pred - target) * mask
    l2 = (pred - target) ** 2 * mask

    denom = mask.sum().clamp(min=1.0) * pred.shape[-1]
    l1 = l1.sum() / denom
    l2 = l2.sum() / denom

    return l1_weight * l1 + l2_weight * l2, l1.detach(), l2.detach()


dataset = ReversePauseRhythmDataset(PAIR_DIR)
n_total = len(dataset)
n_train = int(0.9 * n_total)
n_val = n_total - n_train

train_set, val_set = torch.utils.data.random_split(
    dataset,
    [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(
    train_set,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_reverse
)

val_loader = DataLoader(
    val_set,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_reverse
)

print("Train pairs:", len(train_set))
print("Val pairs:", len(val_set))

In [ ]:
model = ReversePauseRhythmUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

best_val = float("inf")

for epoch in range(1, 21):
    model.train()
    train_loss_sum = 0.0
    n_train_batches = 0

    for batch in train_loader:
        x_t = batch["x_t"].to(DEVICE)
        x_prev = batch["x_prev"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)
        t = batch["t"].to(DEVICE)
        T = batch["T"].to(DEVICE)

        optimizer.zero_grad()
        pred = model(x_t, t, T)
        loss, _, _ = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        n_train_batches += 1

    train_loss = train_loss_sum / max(n_train_batches, 1)

    model.eval()
    val_loss_sum = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for batch in val_loader:
            x_t = batch["x_t"].to(DEVICE)
            x_prev = batch["x_prev"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)
            t = batch["t"].to(DEVICE)
            T = batch["T"].to(DEVICE)

            pred = model(x_t, t, T)
            loss, _, _ = masked_l1_l2_loss(pred, x_prev, mask, l1_weight=1.0, l2_weight=1.0)

            val_loss_sum += loss.item()
            n_val_batches += 1

    val_loss = val_loss_sum / max(n_val_batches, 1)

    print(f"Epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), MODEL_PATH)
        print("  Saved best model to", MODEL_PATH)

In [ ]:
best_model = ReversePauseRhythmUNetLite(dim_in=1024, hidden=384, time_dim=128).to(DEVICE)
best_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
best_model.eval()

print("Loaded:", MODEL_PATH)

In [ ]:
@torch.no_grad()
def reverse_pause_rhythm_diffusion(
    model: nn.Module,
    x_T: np.ndarray,
    T_steps: int,
    device: str = "cpu"
):
    x = torch.from_numpy(x_T.astype(np.float32)).unsqueeze(0).to(device)

    for t_cur in range(T_steps, 0, -1):
        t_tensor = torch.tensor([t_cur], dtype=torch.long, device=device)
        T_tensor = torch.tensor([T_steps], dtype=torch.long, device=device)
        x = model(x, t_tensor, T_tensor)

    return x.squeeze(0).cpu().numpy().astype(np.float32)

In [ ]:
sample_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))
x_clean = np.load(sample_files[0]).astype(np.float32)

x_states, deltas, metas = pause_rhythm_forward_diffusion_chain_pronounced(
    x_clean=x_clean,
    T=T_STEPS,
    schedule=SCHEDULE,
    seed=SEED,
    **PAUSE_RHYTHM_CFG
)
x_T = x_states[-1]

x_recovered = reverse_pause_rhythm_diffusion(
    model=best_model,
    x_T=x_T,
    T_steps=T_STEPS,
    device=DEVICE
)

print("clean -> x_T cosine:", cosine_similarity_mean(x_clean, x_T))
print("clean -> recovered cosine:", cosine_similarity_mean(x_clean, x_recovered))
print("identity MAE:", float(np.mean(np.abs(x_clean - x_T))))
print("recovered MAE:", float(np.mean(np.abs(x_clean - x_recovered))))
print("last meta:", metas[-1])

In [ ]:
test_files = sorted(glob(os.path.join(CLEAN_DIR, "*.npy")))[:20]
all_results = []

for i, fpath in enumerate(test_files, 1):
    x_clean = np.load(fpath).astype(np.float32)

    x_states, deltas, metas = pause_rhythm_forward_diffusion_chain_pronounced(
        x_clean=x_clean,
        T=T_STEPS,
        schedule=SCHEDULE,
        seed=SEED + i,
        **PAUSE_RHYTHM_CFG
    )
    x_T = x_states[-1]

    x_recovered = reverse_pause_rhythm_diffusion(
        model=best_model,
        x_T=x_T,
        T_steps=T_STEPS,
        device=DEVICE
    )

    result = {
        "file": os.path.basename(fpath),
        "clean_to_xT_cos": cosine_similarity_mean(x_clean, x_T),
        "clean_to_rec_cos": cosine_similarity_mean(x_clean, x_recovered),
        "identity_mae": float(np.mean(np.abs(x_clean - x_T))),
        "recovered_mae": float(np.mean(np.abs(x_clean - x_recovered))),
    }
    all_results.append(result)

    print(
        f"[{i:02d}/{len(test_files):02d}] {result['file']} | "
        f"xT cos={result['clean_to_xT_cos']:.6f} | "
        f"rec cos={result['clean_to_rec_cos']:.6f} | "
        f"id mae={result['identity_mae']:.6f} | "
        f"rec mae={result['recovered_mae']:.6f}"
    )

print("\nAverages:")
print("clean -> x_T cosine:", np.mean([r["clean_to_xT_cos"] for r in all_results]))
print("clean -> recovered cosine:", np.mean([r["clean_to_rec_cos"] for r in all_results]))
print("identity MAE:", np.mean([r["identity_mae"] for r in all_results]))
print("recovered MAE:", np.mean([r["recovered_mae"] for r in all_results]))